# Phase 0 — Loader Verification

**Notebook:** `notebooks/00_setup/04_verify_loaders.ipynb`

Quick sanity-check that `src/data/loaders.py` works for all three datasets and that the schema contract holds. Run this after `03_csv_to_parquet.ipynb`.

What it checks:
1. Each loader runs without error
2. Returned shapes, dtypes, contract columns
3. Load times (so we know whether further optimization is needed)
4. Class balance per dataset
5. Day-stream for CICIDS2017

**Note:** if you previously had loaders.py without the categorical-dtype fix, restart the Colab runtime (Runtime → Restart) before running this notebook — otherwise the cached old import will be used.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/phd_thesis')

import time
import numpy as np
import pandas as pd

from src.data.loaders import (
    load_cicids2017, load_unsw_nb15, load_nsl_kdd,
    cicids2017_day_stream, CICIDS2017_DAY_ORDER, DATASET_LOADERS,
)
from src.utils.seeding import set_global_seed

set_global_seed(42)
print('Imports OK.')

Mounted at /content/drive
Imports OK.


## 1. CICIDS2017 — per-day loads

Time each day individually. These are the loads we care most about because Phase 1 streaming experiments call them in sequence.

In [2]:
print(f'{"day":<12}{"rows":>10}{"features":>10}{"benign":>10}{"attack":>10}'
      f'{"load_s":>10}{"mem_MB":>10}')
print('-' * 72)

day_summaries = []
for day in CICIDS2017_DAY_ORDER:
    t0 = time.time()
    try:
        X, y, meta = load_cicids2017(day=day)
    except FileNotFoundError as e:
        print(f'{day:<12}  [MISSING] {e}')
        continue
    load_s = time.time() - t0
    mem_mb = (X.memory_usage(deep=True).sum() + y.memory_usage(deep=True)
              + meta.memory_usage(deep=True).sum()) / 1024**2
    benign = int((y == 0).sum())
    attack = int((y == 1).sum())
    print(f'{day:<12}{len(X):>10}{X.shape[1]:>10}{benign:>10}{attack:>10}'
          f'{load_s:>10.2f}{mem_mb:>10.1f}')
    day_summaries.append({
        'day': day, 'rows': len(X), 'features': X.shape[1],
        'benign': benign, 'attack': attack,
        'load_s': load_s, 'mem_mb': mem_mb,
    })

print()
if day_summaries:
    total_rows = sum(d['rows'] for d in day_summaries)
    total_time = sum(d['load_s'] for d in day_summaries)
    print(f'Total: {total_rows:,} rows in {total_time:.1f}s '
          f'({total_rows/total_time:,.0f} rows/sec)')

day               rows  features    benign    attack    load_s    mem_MB
------------------------------------------------------------------------
monday          371621        82    371621         0      4.14     119.1
tuesday         322078        82    315106      6972      3.98     103.2
wednesday       496640        82    319119    177521      3.51     159.1
thursday        362075        82    288171     73904      2.68     116.0
friday          547557        82    288544    259013      4.49     175.5

Total: 2,099,971 rows in 18.8s (111,709 rows/sec)


## 2. CICIDS2017 — full load (all 5 days)

In [3]:
t0 = time.time()
X, y, meta = load_cicids2017()
load_s = time.time() - t0

print(f'Combined load: {load_s:.2f}s')
print(f'  X     shape: {X.shape}  dtypes: {set(X.dtypes.astype(str).unique())}')
print(f'  y     shape: {y.shape}  dtype:  {y.dtype}  unique: {sorted(y.unique().tolist())}')
print(f'  meta  shape: {meta.shape}  columns: {meta.columns.tolist()}')
print(f'  meta  dtypes: {dict(meta.dtypes.astype(str))}')

print('\nClass balance (overall):')
print(y.value_counts().rename({0: 'benign', 1: 'attack'}).to_string())

print('\nTop 10 attack categories:')
print(meta['attack_category'].value_counts().head(10).to_string())

print(f'\nAttempted flag: True={int(meta["attempted"].sum()):,}, '
      f'False={int((~meta["attempted"]).sum()):,}')

# Use .unique() instead of .cat.categories — works regardless of dtype
print(f'\nSource files: {sorted(meta["source_file"].unique().tolist())}')

# Sanity assertions — these MUST hold or downstream code will break
assert X.isna().sum().sum() == 0, 'Features contain NaN!'
assert y.dtype == np.int8, f'Label dtype is {y.dtype}, expected int8'
assert set(y.unique()) <= {0, 1}, f'Label has unexpected values: {y.unique()}'
assert len(X) == len(y) == len(meta), 'X, y, meta lengths mismatch'
assert isinstance(meta['attack_category'].dtype, pd.CategoricalDtype), \
    'attack_category should be categorical'
assert isinstance(meta['source_file'].dtype, pd.CategoricalDtype), \
    'source_file should be categorical'
print('\n✓ Schema contract holds for CICIDS2017.')

Combined load: 3.95s
  X     shape: (2099971, 82)  dtypes: {'float32', 'int64', 'int32'}
  y     shape: (2099971,)  dtype:  int8  unique: [0, 1]
  meta  shape: (2099971, 3)  columns: ['attack_category', 'attempted', 'source_file']
  meta  dtypes: {'attack_category': 'category', 'attempted': 'bool', 'source_file': 'category'}

Class balance (overall):
label
benign    1582561
attack     517410

Top 10 attack categories:
attack_category
BENIGN                     1582561
Portscan                    159066
DoS Hulk                    159049
DDoS                         95144
Infiltration - Portscan      71767
DoS GoldenEye                 7647
DoS Slowloris                 5706
DoS Slowhttptest              5108
Botnet                        4803
FTP-Patator                   3984

Attempted flag: True=11,979, False=2,087,992

Source files: ['friday', 'monday', 'thursday', 'tuesday', 'wednesday']

✓ Schema contract holds for CICIDS2017.


## 3. CICIDS2017 — day stream

This is the API Phase 1 streaming experiments will use.

In [4]:
print('Streaming days in chronological order:\n')
for day_name, X_d, y_d, meta_d in cicids2017_day_stream():
    bal = y_d.value_counts(normalize=True).to_dict()
    pct_attack = bal.get(1, 0.0) * 100
    top_attack = (meta_d.loc[y_d == 1, 'attack_category']
                  .value_counts().head(1))
    top_str = f'{top_attack.index[0]} ({top_attack.iloc[0]:,})' if len(top_attack) else 'none'
    print(f'  {day_name:<12} rows={len(X_d):>8,}  attack={pct_attack:>5.1f}%  '
          f'top_attack={top_str}')

Streaming days in chronological order:

  monday       rows= 371,621  attack=  0.0%  top_attack=BENIGN (0)
  tuesday      rows= 322,078  attack=  2.2%  top_attack=FTP-Patator (3,984)
  wednesday    rows= 496,640  attack= 35.7%  top_attack=DoS Hulk (159,049)
  thursday     rows= 362,075  attack= 20.4%  top_attack=Infiltration - Portscan (71,767)
  friday       rows= 547,557  attack= 47.3%  top_attack=Portscan (159,066)


## 4. UNSW-NB15

In [5]:
for partition in [None, 'train', 'test']:
    t0 = time.time()
    try:
        X, y, meta = load_unsw_nb15(partition=partition) if partition \
                     else load_unsw_nb15()
    except FileNotFoundError as e:
        print(f'  {str(partition):<10}: [MISSING] {e}')
        continue
    label = partition if partition else 'all'
    print(f'  {label:<10}: rows={len(X):>8,}  cols={X.shape[1]:>3}  '
          f'attack={(y==1).mean()*100:>5.1f}%  load={time.time()-t0:.2f}s')
    # contract check
    assert X.isna().sum().sum() == 0
    assert set(y.unique()) <= {0, 1}

print('\nTop UNSW attack categories:')
X, y, meta = load_unsw_nb15()
print(meta.loc[y == 1, 'attack_category'].value_counts().head(10).to_string())
print('\n✓ Schema contract holds for UNSW-NB15.')

  all       : rows= 257,673  cols= 42  attack= 63.9%  load=2.59s
  train     : rows=  82,332  cols= 42  attack= 55.1%  load=0.08s
  test      : rows= 175,341  cols= 42  attack= 68.1%  load=0.15s

Top UNSW attack categories:
attack_category
Generic           58871
Exploits          44525
Fuzzers           24246
DoS               16353
Reconnaissance    13987
Analysis           2677
Backdoor           2329
Shellcode          1511
Worms               174
Normal                0

✓ Schema contract holds for UNSW-NB15.


## 5. NSL-KDD

In [6]:
for partition in [None, 'train', 'test']:
    t0 = time.time()
    try:
        X, y, meta = load_nsl_kdd(partition=partition) if partition \
                     else load_nsl_kdd()
    except FileNotFoundError as e:
        print(f'  {str(partition):<10}: [MISSING] {e}')
        continue
    label = partition if partition else 'all'
    print(f'  {label:<10}: rows={len(X):>8,}  cols={X.shape[1]:>3}  '
          f'attack={(y==1).mean()*100:>5.1f}%  load={time.time()-t0:.2f}s')
    assert X.isna().sum().sum() == 0
    assert set(y.unique()) <= {0, 1}

print('\nTop NSL-KDD attack categories:')
X, y, meta = load_nsl_kdd()
print(meta.loc[y == 1, 'attack_category'].value_counts().head(10).to_string())
print('\n✓ Schema contract holds for NSL-KDD.')

  all       : rows= 148,517  cols= 41  attack= 48.1%  load=1.76s
  train     : rows= 125,973  cols= 41  attack= 46.5%  load=0.06s
  test      : rows=  22,544  cols= 41  attack= 56.9%  load=0.02s

Top NSL-KDD attack categories:
attack_category
neptune         45871
satan            4368
ipsweep          3740
smurf            3311
portsweep        3088
nmap             1566
back             1315
guess_passwd     1284
mscan             996
warezmaster       964

✓ Schema contract holds for NSL-KDD.


## 6. Generic loader by name

Verifies the registry works — Phase 1+ experiments will use `load_dataset(name)` from configs.

In [7]:
from src.data.loaders import load_dataset

for name in DATASET_LOADERS:
    try:
        X, y, meta = load_dataset(name)
        print(f'  {name:<15}  X={X.shape}  attack_rate={(y==1).mean():.4f}')
    except Exception as e:
        print(f'  {name:<15}  FAILED: {e}')

print('\nAll loaders verified.')

  cicids2017       X=(2099971, 82)  attack_rate=0.2464
  unsw_nb15        X=(257673, 42)  attack_rate=0.6391
  nsl_kdd          X=(148517, 41)  attack_rate=0.4812

All loaders verified.


## 7. Sanity check — feature columns are numeric

Downstream RF/SHAP code requires this. If any feature column is object/string, training will fail with a confusing error.

In [8]:
for name in DATASET_LOADERS:
    X, y, meta = load_dataset(name)
    bad = X.select_dtypes(exclude=['number']).columns.tolist()
    if bad:
        print(f'  ⚠ {name}: non-numeric feature columns: {bad}')
    else:
        print(f'  ✓ {name}: all {X.shape[1]} feature columns are numeric '
              f'(dtypes: {set(X.dtypes.astype(str).unique())})')

  ✓ cicids2017: all 82 feature columns are numeric (dtypes: {'float32', 'int64', 'int32'})
  ✓ unsw_nb15: all 42 feature columns are numeric (dtypes: {'float32', 'int64', 'int32'})
  ✓ nsl_kdd: all 41 feature columns are numeric (dtypes: {'float32', 'int32'})


## Next steps

If everything above shows ✓:

1. Phase 0 infrastructure is complete.
2. Read the Engelen 2021 paper if you haven't yet (see `data_card.md` for the link).
3. Write a 2-page research-direction note for your advisor recapping the pivot from "adaptive IDS" to "explanation-space dynamics under drift." Save to `reports/advisor_notes/`.
4. Start Phase 1 → `notebooks/01_phase0_foundation/05_data_characterization.ipynb` — per-day class distribution, feature stats, day-vs-day distribution shift on CICIDS2017.